In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 — Bronze Ingestion
# MAGIC **Layer:** Bronze | **Catalog:** fintech_lakehouse_dev | **Schema:** transactions
# MAGIC
# MAGIC Ingests raw fintech transaction data from ADLS Gen2 into Delta Bronze tables.
# MAGIC Schema enforcement enabled with `_rescued_data` capture for malformed records.

# COMMAND ----------

from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType

CATALOG = "fintech_lakehouse_dev"
SCHEMA = "transactions"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"

BRONZE_CORE = f"{CATALOG}.{SCHEMA}.bronze_core_banking"
BRONZE_CARD = f"{CATALOG}.{SCHEMA}.bronze_card_auth"

print(f"Target catalog: {CATALOG}")
print(f"Volume path:    {VOLUME_PATH}")

In [0]:
# COMMAND ----------
# Read core banking CSV
core_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")   # all strings — enforce types in Silver
    .option("rescuedDataColumn", "_rescued_data")
    .load(f"{VOLUME_PATH}/core_banking/")
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("bronze_source", lit("core_banking"))
)

print(f"Core banking rows: {core_df.count()}")
core_df.printSchema()

In [0]:
# COMMAND ----------
# Read card authorization CSV
card_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("rescuedDataColumn", "_rescued_data")
    .load(f"{VOLUME_PATH}/card_auth/")
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("bronze_source", lit("card_auth"))
)

print(f"Card auth rows: {card_df.count()}")
card_df.printSchema()

In [0]:
# COMMAND ----------
# Write to Delta Bronze — overwrite for idempotent reruns
core_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(BRONZE_CORE)

card_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(BRONZE_CARD)

print(f"Written → {BRONZE_CORE}")
print(f"Written → {BRONZE_CARD}")

In [0]:
%sql
-- COMMAND ----------
%sql
-- Verify row counts and rescued data
SELECT 'bronze_core_banking' AS table_name, COUNT(*) AS row_count,
       SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) AS rescued_rows
FROM fintech_lakehouse_dev.transactions.bronze_core_banking
UNION ALL
SELECT 'bronze_card_auth', COUNT(*),
       SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END)
FROM fintech_lakehouse_dev.transactions.bronze_card_auth